In [16]:
from dotenv import load_dotenv
from pathlib import Path
import sys
import os

# Walk up until we find the project root (folder with the .env)
current_path = Path().resolve()
for parent in [current_path] + list(current_path.parents):
    if (parent / ".env").exists():
        load_dotenv(parent / ".env")
        project_root = os.getenv("PROJECT_ROOT")
        print(project_root)
        sys.path.append(project_root)     
        break


%load_ext autoreload
%autoreload 2

C:\Users\Admin\Desktop\Emmanuel\beluga-call-pipeline\
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [39]:
import pandas as pd
import glob
import os

# Path to the snippets directory
snippets_dir = "../../data/evaluation_snippets/v1/"

# Get all folders in the snippets directory
snippet_folders = [f for f in os.listdir(snippets_dir) 
                   if os.path.isdir(os.path.join(snippets_dir, f))]

# Read all files from each folder's manual_verification directory
dfs = []
for folder in snippet_folders:
    manual_verification_dir = os.path.join(snippets_dir, folder, "manual_verification")
    
    # Check if manual_verification directory exists
    if not os.path.exists(manual_verification_dir):
        print(f"No manual_verification folder found in {folder}")
        continue
    
    # Get all files in the manual_verification directory
    manual_files = glob.glob(os.path.join(manual_verification_dir, "*"))
    
    # Read each file and add folder name as a column
    for file in manual_files:
        try:
            df = pd.read_csv(file, sep='\t', engine='python')
            df['site'] = folder  # Add folder name as a column
            df["labeled_snippet_dir"] = os.path.join(snippets_dir, folder)
            dfs.append(df)
            print(f"Read {os.path.basename(file)} from {folder}")
        except Exception as e:
            print(f"Could not read {file}: {e}")

# Concatenate all DataFrames into one
if dfs:
    all_manual_df = pd.concat(dfs, ignore_index=True)
    print(f"\nTotal rows: {len(all_manual_df)}")
    print(f"Datasets: {all_manual_df['site'].unique()}")
else:
    print("No data files found")
    all_manual_df = pd.DataFrame()


Read 201359382.170724133858.snippet.selections.txt from BSM_2017
Read 201359382.170724140912.snippet.selections.txt from BSM_2017
Read 201359382.210714080018.snippet.selections_MA.txt from CAC_2021
Read 201359382.210714083053.snippet.selections_MA.txt from CAC_2021
Read 5725.200726190001.snippet.selections_JAA.txt from KAM_2020
Read 5725.200726191905.snippet.selections_JAA.txt from KAM_2020
Read 5725.200726195504.snippet.selections_JAA.txt from KAM_2020

Total rows: 3601
Datasets: ['BSM_2017' 'CAC_2021' 'KAM_2020']


In [40]:
labels_df = all_manual_df.copy()

labels_df = labels_df.drop(columns=["Selection", "View", "Channel", "Low Freq (Hz)", "High Freq (Hz)", "ECHO", "HFPC", "BBPC", "Whistle"])


In [41]:
labels_df = labels_df.rename(columns={"snippet_filename": "labeled_snippet_filename"})

In [42]:
from pipeline.pipeline import get_hydrophone_model
from data_preprocessing.spectrogram.spectrogram_generator import HYDROPHONE_SENSITIVITY

# Apply get_hydrophone_model to the original_filename column for all rows
labels_df["HydrophoneModel"] = labels_df["original_filename"].apply(get_hydrophone_model)
labels_df["HydrophoneSensitivity"] = labels_df["HydrophoneModel"].apply(HYDROPHONE_SENSITIVITY.get_sensitivity)

In [43]:
labels_df["HydrophoneSensitivity"].value_counts()

HydrophoneSensitivity
-172.7    2400
-175.7    1201
Name: count, dtype: int64

In [44]:
import pandas as pd

# Convert snippet_start_time to datetime and add the offset in seconds
labels_df["clip_start_time"] = pd.to_datetime(labels_df["snippet_start_time"]) + pd.to_timedelta(labels_df["Begin Time (s)"], unit='s')
labels_df["clip_end_time"] = pd.to_datetime(labels_df["snippet_start_time"]) + pd.to_timedelta(labels_df["End Time (s)"], unit='s')

In [45]:
labels_df.rename(columns={"site": "Site"}, inplace=True)
labels_df["Site"] = labels_df["Site"].str.split("_").str[0]

In [46]:
labels_df["clip_filename"] = labels_df["Site"] + "_" + labels_df["clip_start_time"].dt.strftime("%Y%m%d_%H%M%S%f").str[:-4] + ".wav"
# Check for duplicates in clip_filename
duplicate_clips = labels_df[labels_df.duplicated("clip_filename", keep=False)]
if not duplicate_clips.empty:
    print("Duplicates found in 'clip_filename':")
    display(duplicate_clips)
else:
    print("No duplicates found in 'clip_filename'.")

No duplicates found in 'clip_filename'.


In [47]:
def set_verif_flags(gt):
    if pd.isna(gt):
        return pd.Series([False, False, False, False])
    gt_str = str(gt)
    if 'a' in gt_str:
        return pd.Series([False, False, False, False])
    return pd.Series([
        'e' in gt_str,  # ECHO_verif
        'b' in gt_str,  # BBPC_verif
        'h' in gt_str,  # HFPC_verif
        'w' in gt_str   # Whislte_verif
    ])

labels_df[["ECHO", "BBPC", "HFPC", "Whistle"]] = labels_df["GROUNDTRUTH"].apply(set_verif_flags)
# Convert ECHO, BBPC, HFPC, Whistle columns to 0/1 integers
labels_df[["ECHO", "BBPC", "HFPC", "Whistle"]] = labels_df[["ECHO", "BBPC", "HFPC", "Whistle"]].astype(int)


In [48]:
labels_df

,Begin Time (s),End Time (s),GROUNDTRUTH,Notes,original_filename,labeled_snippet_filename,snippet_start_time,snippet_start_s,snippet_end_s,context,...,labeled_snippet_dir,HydrophoneModel,HydrophoneSensitivity,clip_start_time,clip_end_time,clip_filename,ECHO,BBPC,HFPC,Whistle
0,0.0,1.0,a,NaN,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:38:58,536,1136,boat_passage,...,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 13:38:58,2017-07-24 13:38:59,BSM_20170724_13385800.wav,0,0,0,0
1,1.0,2.0,e,NaN,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:38:58,536,1136,boat_passage,...,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 13:38:59,2017-07-24 13:39:00,BSM_20170724_13385900.wav,1,0,0,0
2,2.0,3.0,e,NaN,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:38:58,536,1136,boat_passage,...,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 13:39:00,2017-07-24 13:39:01,BSM_20170724_13390000.wav,1,0,0,0
3,3.0,4.0,e,NaN,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:38:58,536,1136,boat_passage,...,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 13:39:01,2017-07-24 13:39:02,BSM_20170724_13390100.wav,1,0,0,0
4,4.0,5.0,e,NaN,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:38:58,536,1136,boat_passage,...,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 13:39:02,2017-07-24 13:39:03,BSM_20170724_13390200.wav,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3596,242.0,243.0,e,NaN,5725.200726185951.wav,5725.200726195504.snippet.wav,2020-07-26 19:55:04,3313,3560,boat_passage,...,../../data/evaluation_snippets/v1/KAM_2020,5725,-175.7,2020-07-26 19:59:06,2020-07-26 19:59:07,KAM_20200726_19590600.wav,1,0,0,0
3597,243.0,244.0,e,NaN,5725.200726185951.wav,5725.200726195504.snippet.wav,2020-07-26 19:55:04,3313,3560,boat_passage,...,../../data/evaluation_snippets/v1/KAM_2020,5725,-175.7,2020-07-26 19:59:07,2020-07-26 19:59:08,KAM_20200726_19590700.wav,1,0,0,0
3598,244.0,245.0,e,NaN,5725.200726185951.wav,5725.200726195504.snippet.wav,2020-07-26 19:55:04,3313,3560,boat_passage,...,../../data/evaluation_snippets/v1/KAM_2020,5725,-175.7,2020-07-26 19:59:08,2020-07-26 19:59:09,KAM_20200726_19590800.wav,1,0,0,0
3599,245.0,246.0,e,NaN,5725.200726185951.wav,5725.200726195504.snippet.wav,2020-07-26 19:55:04,3313,3560,boat_passage,...,../../data/evaluation_snippets/v1/KAM_2020,5725,-175.7,2020-07-26 19:59:09,2020-07-26 19:59:10,KAM_20200726_19590900.wav,1,0,0,0


In [49]:
labels_df["BBPC"].value_counts()

BBPC
0    3393
1     208
Name: count, dtype: int64

## Clipping to 1 second audio files

In [50]:
import os
import librosa
import soundfile as sf
from tqdm import tqdm

# Output directory
output_dir = "../../data/Verified_Dataset/clip_wavs"
os.makedirs(output_dir, exist_ok=True)

# Group by snippet to load each file only once
grouped = labels_df.groupby(["labeled_snippet_dir", "labeled_snippet_filename"])

for (snippet_dir, snippet_filename), group in tqdm(grouped, total=len(grouped)):
    # Build the full path to the source snippet
    source_path = os.path.join(snippet_dir, snippet_filename)
    
    try:
        # Load the entire snippet once
        y, sr = librosa.load(source_path, sr=None)
        
        # Extract all clips from this snippet
        for idx, row in group.iterrows():
            output_path = os.path.join(output_dir, row["clip_filename"])
            
            # Skip if already exists
            if os.path.exists(output_path):
                continue
            
            # Calculate sample indices
            start_sample = int(row["Begin Time (s)"] * sr)
            end_sample = int(row["End Time (s)"] * sr)
            
            # Extract and save the clip
            clip = y[start_sample:end_sample]
            sf.write(output_path, clip, sr)
            
    except Exception as e:
        print(f"Error processing {snippet_filename}: {e}")

100%|██████████| 7/7 [00:01<00:00,  3.53it/s]


In [51]:
labels_df["Boat"] = labels_df["context"].apply(lambda x: 1 if x == "boat_passage" else 0)
labels_df = labels_df.drop(columns=["context"])


In [52]:
labels_df

,Begin Time (s),End Time (s),GROUNDTRUTH,Notes,original_filename,labeled_snippet_filename,snippet_start_time,snippet_start_s,snippet_end_s,Site,...,HydrophoneModel,HydrophoneSensitivity,clip_start_time,clip_end_time,clip_filename,ECHO,BBPC,HFPC,Whistle,Boat
0,0.0,1.0,a,NaN,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:38:58,536,1136,BSM,...,201359382,-172.7,2017-07-24 13:38:58,2017-07-24 13:38:59,BSM_20170724_13385800.wav,0,0,0,0,1
1,1.0,2.0,e,NaN,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:38:58,536,1136,BSM,...,201359382,-172.7,2017-07-24 13:38:59,2017-07-24 13:39:00,BSM_20170724_13385900.wav,1,0,0,0,1
2,2.0,3.0,e,NaN,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:38:58,536,1136,BSM,...,201359382,-172.7,2017-07-24 13:39:00,2017-07-24 13:39:01,BSM_20170724_13390000.wav,1,0,0,0,1
3,3.0,4.0,e,NaN,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:38:58,536,1136,BSM,...,201359382,-172.7,2017-07-24 13:39:01,2017-07-24 13:39:02,BSM_20170724_13390100.wav,1,0,0,0,1
4,4.0,5.0,e,NaN,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:38:58,536,1136,BSM,...,201359382,-172.7,2017-07-24 13:39:02,2017-07-24 13:39:03,BSM_20170724_13390200.wav,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3596,242.0,243.0,e,NaN,5725.200726185951.wav,5725.200726195504.snippet.wav,2020-07-26 19:55:04,3313,3560,KAM,...,5725,-175.7,2020-07-26 19:59:06,2020-07-26 19:59:07,KAM_20200726_19590600.wav,1,0,0,0,1
3597,243.0,244.0,e,NaN,5725.200726185951.wav,5725.200726195504.snippet.wav,2020-07-26 19:55:04,3313,3560,KAM,...,5725,-175.7,2020-07-26 19:59:07,2020-07-26 19:59:08,KAM_20200726_19590700.wav,1,0,0,0,1
3598,244.0,245.0,e,NaN,5725.200726185951.wav,5725.200726195504.snippet.wav,2020-07-26 19:55:04,3313,3560,KAM,...,5725,-175.7,2020-07-26 19:59:08,2020-07-26 19:59:09,KAM_20200726_19590800.wav,1,0,0,0,1
3599,245.0,246.0,e,NaN,5725.200726185951.wav,5725.200726195504.snippet.wav,2020-07-26 19:55:04,3313,3560,KAM,...,5725,-175.7,2020-07-26 19:59:09,2020-07-26 19:59:10,KAM_20200726_19590900.wav,1,0,0,0,1


In [53]:
labels_output_dir = "../../data/Verified_Dataset/labels"

labels_df.to_csv(os.path.join(labels_output_dir, "labels_eval_v1.csv"), index=False)